# Paper Writing Notebook (some technical reasons for different choices)

## Motivation

We have already done some work on binary classification of $\text{Hp}30_{max} \geq 5$ in Billcliff et al. (2025), and wish to expand this approach to a better probabilistic determination for an Hp30 forecast. In general we wish to provide a probability for exceeding some specific thresholds. The baseline idea is that we want to expand from classification to regression, where we are targeting a specific value of the hp30 index within a forecast window, rather than just if we breach the $\text{Hp}30_{max} \geq 5$ threshold. We will frame the problem similar to Bernoux et al. (2022), which forecasts daily $\text{Kp}_{max}$.

## First Approach at $\text{Hp}30_{max} \geq 5$

The same approach as before can be tested (with the only switch being from logistic regression -a classification model- to linear regression -a regression based model). We create 100 ensembles trained on HUXt and aggregate as in Billcliff et al. (2025). Additionally, we continue to compare against baseline models of persistence and 27-day recurrence. A new baseline model "persistence_max" is introduced, with it's forecast given as the maximum value of Hp30 within the 24 hour window preceeding $T_0$ (time that the forecast is made). 

Beating (as in getting better metrics than) persistence, 27-day recurrence, and persistence_max proves to be more challenging for this problem definition. 

1. The baseline models forecasts are more powerful here, because they are making a more specific forecast. In the binary classification problem, each of these baselines makes a deterministic forecast either of class 0 or 1 with no associated probability. This means in hard to forecast events (or non-events) the "weighted_mean" forecast from Billciff et al. (2025) will be less punished by the metrics since it's providing a more reliable probabilistic forecast that we breach the $\text{Hp}30_{max} \geq 5$ threshold.

2. In the new problem, these baselines are more aligned with the naive approach of forecasting a single Hp30 value, and thus provide a much better comparison for our metrics, but increasing the difficulty for outperforming these models.

Below is the linear regression ensemble as an additional baseline, showing that it performs worse for this regression problem. 

<img src="../../storm_utils/figures/data_plots/regression-Forecast-MAE_model_vs_lead_time.png" 
     alt="First Look" width="500">

<img src="../../storm_utils/figures/data_plots/regression-Forecast-RMSE_model_vs_lead_time.png" 
     alt="First Look" width="500">

<i><b>Figure 1.</b> MAE and RMSE for 3 baseline models compared with the adaptation of the aggregated ensemble of linear regressions. Metrics are shown at progressively increasing lead times (along the x-axis).</i> 

<b>Figure 1</b> shows that our model is worse than the baselines. In terms of MAE, we beat persistence (slightly) at lead times of 1, 3, 6 hours. The reason we may beat persistence in MAE, but not RMSE is that the weighted_mean is less consistent, i.e, sometimes it gets really close to the observed values and sometimes it is quite far off, reflected as a reasonable MAE but terrible RMSE (because of squaring the errors in the worst cases).

We can also take a look at the distribution of our forecasts to see if we are, in general, over- or under-predicting

<img src="../../storm_utils/figures/data_plots/forecast_distribution.png" 
     alt="Distribution" width="500">

<i><b>Figure 2.</b> Distribution of storm-time-only test set (top panel) and set of forecasts at 12 hour lead time.</i>

The distribution of forecasts shows that we are consistently underpredicting storm times. Underprediction will likely be a recurring theme of this project... 

Now, we move on to discussing distribution in a different way. Rather than thinking of distribution as above, we will instead forecast a range of values that we may expect to see for a given event (forecasted as a distribution around a central point). 

## Distribution Forecasting

A great way to extract uncertainties / probabilites on a forecast when we wish to forecast a single value is through the use of a distribution forecast. Essentially, we pick some standard distribution which can be described by a few parameters, and try to forecast what these parameters are. In the simplist case, we could forecast a normal (gaussian) distribution by forecasting two parameters: mean <code>&mu;</code> and deviation from the mean <code>&sigma;</code>. This results in an estimation of the forecast as a distribution: $N(\mu, \sigma)$

In practice, the forecast of for the event would be <code>&mu;</code>. We then use <code>&sigma;</code> to estimate the confidence of the forecast to get uncertainty. In a similar vain, we could reframe this approach as a "likelihood that we exceed a given threshold" and use the <b>Cumuluative Distribution Function (CDF)</b> of the normal distribution to extract probabilities that we exceed each threshold of Hp30. The <b>CDF, F(x)</b> is the area under the <b>PDF, f(x)</b> curve as we vary <code>x</code>. 

### The Normal Distribution

As described above, we can use the normal distribution $N(\mu, \sigma)$ as a first approach for distribution forecasting. 

<b>Probability Density Function (PDF)</b> for $N(\mu, \sigma)$ is 

$$
f(x) = \frac{1}{\sqrt{2\pi\sigma^2}}\exp\!\left({-\frac{(x-\mu)^2}{2\sigma^2}}\right)\
$$

and <b>Cumulative Distribution Function (CDF)</b> is

$$
F_X(x) \;=\; \Pr(X \le x)
\;=\; \frac{1}{\sigma\sqrt{2\pi}}\int_{-\infty}^{x}
\exp\!\left(-\frac{(t-\mu)^2}{2\sigma^2}\right)\,dt.
$$

It helps to plot these to see what happens when we change $\mu$ and $\sigma$. 

#### Interactive Plot Code

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.special import erf
from ipywidgets import interactive, FloatSlider, VBox

# Normal PDF
def normal_pdf(x, mu, sigma):
    return (1/(sigma * np.sqrt(2*np.pi))) * np.exp(-0.5*((x - mu)/sigma)**2)

# Normal CDF
def normal_cdf(x, mu, sigma):
    return 0.5 * (1 + erf((x - mu) / (sigma*np.sqrt(2))))

def plot_normal(mu, sigma, x_max=10.0, x0=0.0):
    x = np.linspace(mu - x_max, mu + x_max, 1000)

    y_pdf = normal_pdf(x, mu, sigma)
    y_cdf = normal_cdf(x, mu, sigma)
    p_exceeds = 1 - normal_cdf(x0, mu, sigma)

    plt.figure(figsize=(10, 5))

    # PDF
    plt.subplot(1, 2, 1)
    plt.plot(x, y_pdf, lw=2)
    plt.axvline(x0, linestyle="--", lw=1)
    plt.title(f"Normal PDF — μ={mu:.2f}, σ={sigma:.2f}")
    plt.xlabel("x")
    plt.ylabel("PDF")
    plt.grid(alpha=0.3)

    # CDF
    plt.subplot(1, 2, 2)
    plt.plot(x, y_cdf, lw=2)
    plt.axvline(x0, linestyle="--", lw=1)
    plt.title(
        f"Normal CDF — μ={mu:.2f}, σ={sigma:.2f}\n"
        f"P(X > {x0:.2f}) = {p_exceeds:.4f}"
    )
    plt.xlabel("x")
    plt.ylabel("CDF")
    plt.ylim(0, 1)
    plt.grid(alpha=0.3)

    plt.tight_layout()
    plt.show()


# Sliders
mu_slider     = FloatSlider(value=0.0, min=-5.0, max=5.0, step=0.1, description='μ')
sigma_slider  = FloatSlider(value=1.0, min=0.1, max=5.0, step=0.05, description='σ')
xmax_slider   = FloatSlider(value=5.0, min=2.0, max=15.0, step=0.5, description='x max')
x0_slider     = FloatSlider(value=0.0, min=-5.0, max=5.0, step=0.1, description='threshold')

normal = interactive(plot_normal, mu=mu_slider, sigma=sigma_slider,
                x_max=xmax_slider, x0=x0_slider)

#### Interactive Plot

In [ ]:
display(VBox([normal]))

<b>Figure 3.</b> Interactive plot for parameter effect on the Normal Distribution.

### The Weibull Distribution

We know that the normal distribution is symmetric and consequently, a forecast for <code>&mu;</code> and <code>&sigma;</code> will be symmetrical either side of the mean <code>&mu;</code>. In practice this isn't ideal since we'd like to have some sort of confidence skewed in one direction or another. For example, a model may forecast expected $Hp30_{max}$ and be pretty confident that $Hp30$ will reach that value, but not know to what extend $Hp30$ will exceed this threshold by. This deeper kind of analytics and model insight requires a different distribution, so we call upon the <strong>Weibull distribution</strong> as an alternative approach for distribution forecasting with an asymmetric (or skewed) distribution. 

The <strong>Probability Density Function (PDF)</strong> for a Weibull distribution with <em>shape parameter</em> <code>k</code> and <em>scale parameter</em> <code>&lambda;</code> is:


$$
f(x) = \frac{k}{\lambda} \left(\frac{x}{\lambda}\right)^{k-1} \exp\left[-\left(\frac{x}{\lambda}\right)^k\right], \quad x \ge 0
$$


and the <strong>Cumulative Distribution Function (CDF)</strong> is:


$$
F_X(x) = \Pr(X \le x) = 1 - \exp\left[-\left(\frac{x}{\lambda}\right)^k\right], \quad x \ge 0.
$$

It helps to plot these functions to see how the distribution changes when we vary the shape <code>k</code> and scale <code>&lambda;</code> parameters.


#### Interactive Plot Code

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interactive, FloatSlider, VBox

# Weibull PDF
def weibull_pdf(x, k, lam):
    return (k / lam) * (x / lam)**(k - 1) * np.exp(-(x / lam)**k)

# Weibull CDF
def weibull_cdf(x, k, lam):
    return 1 - np.exp(-(x / lam)**k)

def plot_weibull_separate(lam, k, x_max=10.0, x0=5.0):
    x = np.linspace(0, x_max, 1000)

    pdf = weibull_pdf(x, k, lam)
    cdf = weibull_cdf(x, k, lam)
    prob_exceed = np.exp(-(x0 / lam)**k)

    plt.figure(figsize=(10, 5))

    # PDF subplot
    plt.subplot(1, 2, 1)
    plt.plot(x, pdf, lw=2)
    plt.axvline(x0, linestyle="--", lw=1)
    plt.title(f"Weibull PDF — λ={lam:.3g}, k={k:.3g}")
    plt.xlabel("x")
    plt.ylabel("PDF")
    plt.grid(alpha=0.3)

    # CDF subplot
    plt.subplot(1, 2, 2)
    plt.plot(x, cdf, lw=2)
    plt.axvline(x0, linestyle="--", lw=1)
    plt.title(f"Weibull CDF — P(X > {x0:.2f}) = {prob_exceed:.4f}")
    plt.xlabel("x")
    plt.ylabel("CDF")
    plt.ylim(0, 1)
    plt.grid(alpha=0.3)

    plt.tight_layout()
    plt.show()


# Sliders
lam_slider = FloatSlider(value=2.0, min=0.1, max=15.0, step=0.01,
                         description='λ (scale)', continuous_update=True)
k_slider   = FloatSlider(value=2.0, min=0.1, max=10.0, step=0.01,
                         description='k (shape)', continuous_update=True)
xmax_slider = FloatSlider(value=10.0, min=5.0, max=30.0, step=0.5,
                          description='x max')
x0_slider   = FloatSlider(value=5.0, min=0.1, max=20.0, step=0.1,
                          description='threshold')

weibull = interactive(plot_weibull_separate,
                lam=lam_slider, k=k_slider,
                x_max=xmax_slider, x0=x0_slider)


#### Interactive Plot

In [ ]:
display(VBox([weibull]))

<b>Figure 4.</b> Interactive plot for parameter effect on the Weibull Distribution.

We can play about with the plot above and see how the shapes of the CDF changes. <blockquote>Reminder: The <b>CDF</b> tells us likelihood of reaching some specified value of our variable.</blockquote>

As we alter <code>&lambda;</code>, our median forecast value increases. 
As we alter <code>k</code>, the skew of the distribution changes which manifests physically as confidence of the distribution. As <code>k</code> $\rightarrow \infty$, the distribution tends towards an <i>infinite point mass</i> at x = <code>&lambda;</code>, i.e., we are 100% confident that the exact value of the target variable will be <code>&lambda;</code>.

## First Results

The final forecast method is likely going to be some sort of NN that forecasts the two parameters lambda and k for the distribution directly, with something in the loss function to make it train properly. 

Initially, we will take the array of forecasts from the linear regression models and instea of aggregating with a weighted mean, we will just directly fit these values to a weibull distribution for an estimate of <code>&lambda;</code> and <code>k</code>. This leads to a distribution rather than an actual forecast of the target variable so we cannot calculate metrics directly. Instead, we take an average of the distribution (the median) which we can use to calculate metrics.

<b>Figure 5</b> shows our first results for this approach. It has median metrics at a variety of lead times for different models over our whole test set (both storm and non-storm), with <b>Figure 6</b> presenting the metrics for storm times only. 

<img src="../../storm_utils/figures/data_plots/regression-Forecast-RMSE_model_vs_lead_time_balanced.png" 
    alt="Distribution" width="600">
<img src="../../storm_utils/figures/data_plots/regression-Forecast-MAE_model_vs_lead_time_balanced.png" 
    alt="Distribution" width="600">

<b>Figure 5.</b> RMSE and MAE for a variety of models at varying lead times. Tested on the whole balanced dataset. 

<img src="../../storm_utils/figures/data_plots/regression-Forecast-RMSE_model_vs_lead_time_storm_only.png"
    alt="Storm Only" width="600">

<img src="../../storm_utils/figures/data_plots/regression-Forecast-MAE_model_vs_lead_time_storm_only.png" 
    alt="Distribution" width="600">



<b>Figure 6.</b> RMSE and MAE for a variety of models at varying lead times. Tested only on storm times 

This is a pretty decent start - we see that forecasting a distribution and taking median gives us an improvement on our model metrics. However, we still have a bit of a way to go to beat the metrics from Bernoux et al. (2022) shown in <b>Figure 7.</b>

<img src="../../storm_utils/figures/data_plots/Bernoux_metrics.png" 
     alt="Bernoux Metrics" width="500">

## Next step is to add in some other variables, and do some research into which one's we think might be helpful

We look to literature to see which variables have been used in the past for this sort of problem. 

<b>Tan et al. (2018)</b> uses ACE satellite data, including both solar wind and IMF data. In particular, the candidate variables are: 

| Variable | Definition                           
| ---------|--------------------------------------
| Bx       | IMF x component
| By       | IMF y component
| Bz       | IMF z component
|\|B\|      | Total magnetic field strength of IMF
| V        | Solar wind flow speed
| D        | Proton Density
| Kp       | Historical Kp 

Some additional indices / coupling functions are also used, and we take a look at them now.

### Boyle's Index

We define <b>IMF angle</b> with $\theta_T = atan2(B_y, B_z) \in (0^\circ, 180^\circ)$. The $atan2$ function returns "what direction does this vector point?", rather than the slope between the two vectors.

The Boyle's Index (BI), which is calculated with $V, B, \theta$ : $BI = 10^{-4} V^2 + 11.7 B sin^3(\theta/2)$, with $V$: solar wind flow speed, $B_T = \sqrt{B_y^2 + B_z^2}$, and $\theta$ as above.

Boyle's Index is a coupling function between solar wind-magnetosphere. The idea of this specific coupling funnction is to estimate cross-polar cap potential (CPCP), which is a key driver of Geomagnetic Activity, and hence a driver of Hp30 index.  CPCP is the electric potential difference between the dawn and dusk sides of the polar ionosphere, hence measured in kilovolts (kV).

<b> How will we utilise Boyle's index?</b> Given that Boyle's index is calculated using solar wind speed, we can calculate this for each member of the ensemble (during the input window), this way we can provide a unique estimate of Boyle's index to each LinearRegression classifier. Alternatively, we can test if providing the same Boyle's index at each time step calculated with v_omni to each LinearRegression works similarly.  

Values of Boyle's index linked to geomagnetic storms is roughly 

| Boyle Index (kV) | Activity level        | Physical meaning                                |
| ---------------- | --------------------- | ----------------------------------------------- |
| **< 30 kV**      | Quiet                 | Weak solar wind–magnetosphere coupling          |
| **30–60 kV**     | Moderate              | Enhanced convection, substorm activity          |
| **60–100 kV**    | Active                | Strong coupling, sustained geomagnetic activity |
| **> 100 kV**     | **High / storm-time** | Intense storms, strong polar cap convection     |
| **> 150–200 kV** | Extreme               | Major geomagnetic storms (often CME-driven)     |


### Newell Function

The Newell coupling function, also known as dΦ_MP/dt, is a solar wind-magnetosphere coupling function that estimates the rate at which magnetic flux is opened at the magnetopause. Developed by Newell et al. (2007), it emerged from an empirical study of 10 different magnetospheric state variables across multiple solar cycles.

**Formula:**

$$d\Phi_{MP}/dt = v^{4/3} B_T^{2/3} \sin^{8/3}(\theta_c/2)$$

where:
- $v$ is the solar wind velocity (km/s)
- $B_T = \sqrt{B_y^2 + B_z^2}$ is the transverse IMF magnitude (nT)
- $\theta_c = \arctan2(B_y, B_z)$ is the IMF clock angle

**Physical Interpretation:**

The function represents four physical factors controlling magnetopause reconnection:

1. **Rate of field line approach** ($v$): How quickly IMF field lines are convected toward the magnetopause
2. **Merging probability** ($\sin^{8/3}(\theta_c/2)$): The fraction of impacting field lines that actually merge, depending on magnetic shear
3. **Flux opened** ($B_T$): The strength of the IMF determines the amount of flux opened
4. **Merging line length** ($(B_{MP}/B_T)^{1/3}$, simplified using pressure balance): Based on flux matching between the dipolar magnetosphere and uniform IMF

### Metrics for the effect of the OMNI variables: 

<img src="../../storm_utils/figures/data_plots/weibull_median-Forecast-RMSE_OMNI_vars_vs_lead_time_balanced.png" alt="Storm Only" width="600">

<img src="../../storm_utils/figures/data_plots/weibull_median-Forecast-MAE_OMNI_vars_vs_lead_time_balanced.png" width="600">



Much to my surprise, adding V_sw as a parameter actually improved model performance? Ambient solar wind is a key driver for geomagnetic storms, but providing OMNI solar wind flow speed as a hidden parameter within vomni_{i} isn't quite as good as providing the time series explicitly. 

This analysis is slightly incomplete - Boyle had incorrect values due to $\theta_c$ mapping to [-$\pi, \pi$], rather than [0, $\pi$]

### All OMNI Vars 

We test what happens when we use all OMNI variables. Some unusual results were found - along with a strange issue of drastically bad results for some lead times for specific variables. Some further analysis needs to be done on what's happening in these cases. 

On rerunning, the metrics appear to be just fine again - I dont know if it's an issue with memory then? 

Anyway, here are the top 10, and bottom 10 OMNI vars for training the Weibull Median model. 


<b> Top 10 Best </b>

<img src="../../storm_utils/figures/data_plots/weibull_median-Forecast-RMSE_OMNI_vars_vs_lead_time_balanced_run_2026-01-26_15-50-37_metrics.csv.png" alt="Storm Only" width="600">

<img src="../../storm_utils/figures/data_plots/weibull_median-Forecast-MAE_OMNI_vars_vs_lead_time_balanced_run_2026-01-26_15-50-37_metrics.csv.png" width="600">

<b> Top 10 Worst </b>

<img src="../../storm_utils/figures/data_plots/worst_weibull_median-Forecast-RMSE_OMNI_vars_vs_lead_time_balanced_run_2026-01-26_15-50-37_metrics.csv.png" alt="Storm Only" width="600">

<img src="../../storm_utils/figures/data_plots/worst_weibull_median-Forecast-MAE_OMNI_vars_vs_lead_time_balanced_run_2026-01-26_15-50-37_metrics.csv.png" width="600">

As we can see, these worst metrics, aren't exactly the worst - but they have outliers.

After some further analysis, we discovered that the outliers are a result of the LinearRegression models containing extremely large forecasts occasionally. 

### Quick Fix: Forcing the output

For now, we can just force the output of the individual ensembles to between some reasonable range such as <code>[0.01, 15]</code> using <code>np.clip(forecasts, a_min=0.01, a_max=15)</code>

## Moving away from Weibull

The Weibull distribution is actually probably not a great choice for this problem, as it is an optimised distribution for events occuring over time. Because of this we will experiment with the following: 

1. Normal Distribution
2. Lognormal Distribution
3. Sum of lognormals, (3 parameter forecasts)
    * $\mu$ - scale parameter
    * $\sigma$ - width parameter
    * $\Pi$ - contribution parameter

## Changing the underlying ensemble regression

The distrubtion of the ensemble of forecasts will determine the goodness of the fit. One issue I've been having is creating a decent forecast for the underlying ensemble - and this should become the main focus of the work for now. We will test the following (ever expanding list of) methods, paired with some (currently unimportant) distribution fit. 

    1. LinearRegression()
    2. RidgeRegression()
    3. DecisionTree()
    4. RandomForest()
    5. XGBoost()
    6. Flattened MLP

Analysis for the forecasts will come under the following categories: 

    1. Metrics taken from the mean of the distribution (MAE, RMSE, R)
    2. Metrics on the goodness of the distribution (CRPS)
    3. Goodness of fit to the distribution.
    4. General report of under/overpredicting
    5. Case studies

### 1. Metrics from the mean of the distribution

<img width=700 src='../../storm_regression/figures/regression_mae_model_vs_aggregator_lt12_balanced_combined.png'>

Initially, we see that RandomForest and Ridge are immediate improvements over the LinearRegression (about 5% improvement). A note on <b>training times</b> is that for a single training set, 100 Ridge and Linear Regression will train in roughly 20s, while RandomForest takes 9mins (27 times longer...). But in terms of raw metrics, RandomForest is our best bet. A consequence of this training time, is that it not necessarily feasible anymore to test a wide variety of random seeds and test folds and take a median of the metrics. Additionally, it appears the fit becomes less important as the model complexity is increasing. These values can be improved with additional OMNI parameters, but we ignore this analysis for now. 

Also - maybe good again to just point out that we are still outperforming the baselines significantly. 

Perhaps we are reaching the limit of how much we are making use of the ensemble and some work needs to be done in moving to a more complex method in terms of MLPs or NNs in general. 

### 2. Metrics on the goodness of the distribution (CRPS)

<img width=700 src='../../storm_regression/src/figures/regression_mean_ks_model_vs_aggregator_lt12_balanced_combined.png'>

### 3. Goodness of fit to the distribution

<img width=700 src='../../storm_regression/src/figures/regression_crps_model_vs_aggregator_lt12_balanced_combined.png'>